In [8]:
import numpy as np
import matplotlib.pyplot as plt
import datasets
from privacy_estimates.experiments.aml import JobList
from sklearn.metrics import roc_curve, roc_auc_score, auc

In [9]:
from datetime import datetime

def compute_performance(scores_members, scores_non_members):
    mia_performance = {}
    
    member_vals = [val for val in scores_members]
    non_member_vals = [val for val in scores_non_members]
    mia_performance['auc'] = roc_auc_score([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    fpr, tpr, thresholds = roc_curve([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    for target_fpr in (0.01, 0.05, 0.1):
        mia_performance[f'tpr_at_{target_fpr}'] = np.interp(target_fpr, fpr, tpr)
    print(f"AUC: {mia_performance['auc']}, TPR@0.01: {mia_performance['tpr_at_0.01']}, TPR@0.05: {mia_performance['tpr_at_0.05']}, TPR@0.1: {mia_performance['tpr_at_0.1']}")

    # also add the curves
    mia_performance['fpr'] = fpr
    mia_performance['tpr'] = tpr

    return mia_performance

def compute_performance_from_url(url, job_name = None):
    jobs = JobList.from_urls([url])
    if job_name is None:
        job_name = str(datetime.now())
    
    if not os.path.exists(f'./mia_results/{job_name}'):
        test = jobs[0].get_node('estimate_privacy').download_input('scores', f'./mia_results/{job_name}/scores')
        test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', f'./mia_results/{job_name}/challenge_bits')
    scores = datasets.load_from_disk(f'./mia_results/{job_name}/scores')
    bits = datasets.load_from_disk(f'./mia_results/{job_name}/challenge_bits')
    
    membership_scores = np.array([k['score'] for k in scores])
    membership_labels = np.array([k['challenge_bit'] for k in bits])
    members = membership_scores[membership_labels == 1]
    non_members = membership_scores[membership_labels == 0]
    return compute_performance(members, non_members)

## Lets do it for synthetic multiple

In [10]:
import re

def extract_aml_urls(log_file_path):
    # Initialize a list to store the extracted URLs
    aml_urls = []

    # Define a regular expression to match the log entries containing AML URLs
    aml_url_pattern = re.compile(r'AML URL: (https://ml\.azure\.com/runs/[\w\-\?&=/]+)')

    # Open and read the log file
    with open(log_file_path, 'r') as file:
        for line in file:
            match = aml_url_pattern.search(line)
            if match:
                aml_urls.append(match.group(1))

    return aml_urls

In [11]:
mia_methods=("jaccard_25", "embedding_25", "ngram_2")
multiples = (1, 2, 4, 8)

## (1) Let's first do sst-2

In [ ]:
DATASET = 'sst2'

# Extract the AML URLs
aml_urls = extract_aml_urls(f'../job_launch_outputs/synthetic_multiples_all_mias_{DATASET}.txt')
aml_urls[2] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/modest_curtain_tmzj93mm5x?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[5] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/e6c108af-7ba9-47b5-a107-4217612cdfc0?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[8] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/6d83cf55-0316-4343-bb34-4ca1b72a7f8c?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[11] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/28824f36-cb7c-470e-8cae-0fda0e99d724?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls

In [ ]:
method_to_aucs = {}

i = 0
for j, multiple in enumerate(multiples):
    for method in mia_methods:
        job_name = f'multiple_{multiple}_synthetic_{method}'
        print(f"Processing {job_name}")
        performance = compute_performance_from_url(aml_urls[i])
        if j == 0:
            method_to_aucs[method] = [performance['auc']]
        else:
            method_to_aucs[method].append(performance['auc'])
        i += 1

In [ ]:
method_to_aucs

In [ ]:
methods = ['ngram_2', 'jaccard_25', 'embedding_25']
labels = ['Synthetic (2-gram)', r'Synthetic ($\text{SIM}_{jac}$ - $k=25$)', r'Synthetic ($\text{SIM}_{emb}$ - $k=25$)']
colors = ['darkorange', 'darkgreen', 'darkred']

plt.figure(figsize=(6, 6))
plt.axhline(y=0.998, color='darkblue', linestyle='--', linewidth = 2, alpha=1, label = 'Model')
for i, method in enumerate(methods):
    plt.plot([1, 2, 4, 8], method_to_aucs[method], '-o', alpha=0.8, label=labels[i], color = colors[i],
              markersize=10, linewidth=2, markeredgewidth=2, markeredgecolor='white')
plt.xscale('log', base=2)
plt.xlabel(r'Synthetic multiple $m$', fontsize=15)
plt.ylabel('MIA AUC', fontsize=15)
plt.grid(True, which="both", ls="--", alpha=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', alpha=1.0, label = 'Random guess baseline')
plt.legend(fontsize=12)
plt.ylim(0.45, 1.02)
plt.savefig(f'figures/synthetic_multiple_{DATASET}.pdf', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

## (2) Let's repeat this for AgNews

In [ ]:
DATASET = 'agnews'

# Extract the AML URLs
aml_urls = extract_aml_urls(f'../job_launch_outputs/synthetic_multiples_all_mias_{DATASET}.txt')
aml_urls[2] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/blue_spoon_dbctg32fxf?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[5] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/b0cfb064-2a8a-488a-b013-1cf459d8192f?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[8] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/7964ae30-ed33-4fd3-917a-d478a45e7da9?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls[11] = "https://ml.azure.com/experiments/id/cbd45cd3-4fd8-4922-b82a-124527cc98ee/runs/45493830-e5fc-4769-bdf2-82c28cfb1b81?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourceGroups/PPML/providers/Microsoft.MachineLearningServices/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47"
aml_urls

In [ ]:
method_to_aucs = {}

i = 0
for j, multiple in enumerate(multiples):
    for method in mia_methods:
        job_name = f'multiple_{multiple}_synthetic_{method}'
        print(f"Processing {job_name}")
        performance = compute_performance_from_url(aml_urls[i])
        if j == 0:
            method_to_aucs[method] = [performance['auc']]
        else:
            method_to_aucs[method].append(performance['auc'])
        i += 1

In [ ]:
methods = ['ngram_2', 'jaccard_25', 'embedding_25']
labels = ['Synthetic (2-gram)', r'Synthetic ($\text{SIM}_{jac}$ - $k=25$)', r'Synthetic ($\text{SIM}_{emb}$ - $k=25$)']
colors = ['darkorange', 'darkgreen', 'darkred']

plt.figure(figsize=(6, 6))
plt.axhline(y=0.998, color='darkblue', linestyle='--', linewidth = 2, alpha=1, label = 'Model')
for i, method in enumerate(methods):
    plt.plot([1, 2, 4, 8], method_to_aucs[method], '-o', alpha=0.8, label=labels[i], color = colors[i],
              markersize=10, linewidth=2, markeredgewidth=2, markeredgecolor='white')
plt.xscale('log', base=2)
plt.xlabel(r'Synthetic multiple $m$', fontsize=15)
plt.ylabel('MIA AUC', fontsize=15)
plt.grid(True, which="both", ls="--", alpha=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', alpha=1.0, label = 'Random guess baseline')
plt.legend(fontsize=12)
plt.ylim(0.45, 1.02)
plt.savefig(f'figures/synthetic_multiple_{DATASET}.pdf', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()